In [37]:
import pandas as pd
import numpy as np
from pathlib import Path

TRUSTED_PATH = Path("../data/trusted_silver")

df_bairros = pd.read_parquet(TRUSTED_PATH / "bairros.parquet")
df_conc = pd.read_parquet(TRUSTED_PATH / "concorrentes.parquet")
df_eventos = pd.read_parquet(TRUSTED_PATH / "eventos.parquet")
df_pop = pd.read_parquet(TRUSTED_PATH / "populacao.parquet")

ENRIQUECIMENTO TEMPORAL (EVENTOS)

In [38]:
# Garantir datetime
df_eventos["datetime"] = pd.to_datetime(df_eventos["datetime"])

# Extrair data
df_eventos["data"] = df_eventos["datetime"].dt.date

# Dia da semana
df_eventos["dia_semana"] = df_eventos["datetime"].dt.day_name()

# Hora
df_eventos["hora"] = df_eventos["datetime"].dt.hour

# Classificação de período
def classificar_periodo(hora):
    if 5 <= hora < 12:
        return "manha"
    elif 12 <= hora < 18:
        return "tarde"
    else:
        return "noite"

df_eventos["periodo"] = df_eventos["hora"].apply(classificar_periodo)

In [39]:
# Garantir datetime
df_eventos["datetime"] = pd.to_datetime(df_eventos["datetime"])

# Criar coluna data
df_eventos["data"] = df_eventos["datetime"].dt.date

# Criar dia da semana
df_eventos["dia_semana"] = df_eventos["datetime"].dt.day_name()

# Criar hora
df_eventos["hora"] = df_eventos["datetime"].dt.hour

# Classificação período
def classificar_periodo(hora):
    if 5 <= hora < 12:
        return "manha"
    elif 12 <= hora < 18:
        return "tarde"
    else:
        return "noite"

df_eventos["periodo"] = df_eventos["hora"].apply(classificar_periodo)

FLUXO DIÁRIO POR CONCORRENTE

In [40]:
fluxo_diario = (
    df_eventos
    .groupby(["codigo_concorrente", "data", "dia_semana", "periodo"])
    .size()
    .reset_index(name="total_eventos_dia")
)
fluxo_diario

,codigo_concorrente,data,dia_semana,periodo,total_eventos_dia
0,101025009987066,2017-06-26,Monday,noite,1
1,101025009987066,2017-06-30,Friday,noite,3
2,101025009987066,2017-07-01,Saturday,manha,4
3,101025009987066,2017-07-01,Saturday,noite,11
4,101025009987066,2017-07-01,Saturday,tarde,4
...,...,...,...,...,...
40277,1897386380476876,2017-07-27,Thursday,manha,1
40278,1897386380476876,2017-07-28,Friday,noite,1
40279,1897386380476876,2017-07-29,Saturday,manha,2
40280,1897386380476876,2017-07-29,Saturday,noite,1


MÉTRICAS DE FLUXO (MÉDIA, MAX, MIN)

In [41]:
fluxo_analitico = (
    fluxo_diario
    .groupby(["codigo_concorrente", "dia_semana", "periodo"])
    .agg(
        media_fluxo=("total_eventos_dia", "mean"),
        max_fluxo=("total_eventos_dia", "max"),
        min_fluxo=("total_eventos_dia", "min"),
        total_eventos=("total_eventos_dia", "sum")
    )
    .reset_index()
)
fluxo_analitico

,codigo_concorrente,dia_semana,periodo,media_fluxo,max_fluxo,min_fluxo,total_eventos
0,101025009987066,Friday,manha,4.000000,6,3,16
1,101025009987066,Friday,noite,2.200000,3,1,11
2,101025009987066,Friday,tarde,5.250000,9,3,21
3,101025009987066,Monday,manha,3.600000,5,2,18
4,101025009987066,Monday,noite,2.600000,4,1,13
...,...,...,...,...,...,...,...
12267,1897386380476876,Thursday,tarde,1.333333,2,1,4
12268,1897386380476876,Tuesday,noite,1.000000,1,1,1
12269,1897386380476876,Tuesday,tarde,1.000000,1,1,3
12270,1897386380476876,Wednesday,noite,1.000000,1,1,2


FAIXA DE PREÇO DOS CONCORRENTES

In [42]:
faixa_preco = df_conc[[
    "codigo",
    "nome",
    "categoria",
    "faixa_preco"
]].rename(columns={"codigo": "codigo_concorrente"})
faixa_preco

,codigo_concorrente,nome,categoria,faixa_preco
0,431962533652067,Boizão Lanches,Bar,2
1,1663855903830869,Bar do Serjão,Bar,0
2,567824576564110,Recanto Do Kuca,Restaurant,0
3,202740866540615,Dedé Abelhuda,Grocery Store,0
4,1784900838394305,Tenshi Sushi Boteco Itu,Sushi Restaurant,3
...,...,...,...,...
4197,1831179237099774,Supermercado Santana,"Grocery Store, Food & Beverage Company, Market",0
4198,902572199806965,M&S bolos e doces.,Grocery Store,0
4199,1594553520828717,Nai Cakes,"Cupcake Shop, Tea Room",0
4200,1754208714805968,Flamel,"Dessert Shop, Grocery Store",0


GEOGRAFIA + DENSIDADE DEMOGRÁFICA

In [47]:
import numpy as np

# ==========================================
# GEOGRAFIA + DENSIDADE DEMOGRÁFICA
# ==========================================

# 1️⃣ Padronizar chaves
df_conc = df_conc.rename(columns={"codigo": "codigo_concorrente"})
df_bairros = df_bairros.rename(columns={"codigo": "codigo_bairro"})
df_pop = df_pop.rename(columns={"codigo": "codigo_bairro"})

# 2️⃣ Garantir tipos
df_conc["codigo_bairro"] = df_conc["codigo_bairro"].astype("Int64")
df_bairros["codigo_bairro"] = df_bairros["codigo_bairro"].astype("Int64")
df_pop["codigo_bairro"] = df_pop["codigo_bairro"].astype("Int64")

df_pop["populacao"] = df_pop["populacao"].astype("Int64")
df_bairros["area"] = df_bairros["area"].astype("float64")

# 3️⃣ Merge concorrentes + bairros
geo = df_conc.merge(
    df_bairros,
    on="codigo_bairro",
    how="left",
    validate="m:1",
    suffixes=("_conc", "_bairro")
)

# 4️⃣ Merge com população
geo = geo.merge(
    df_pop,
    on="codigo_bairro",
    how="left",
    validate="m:1"
)

# 5️⃣ Cálculo seguro da densidade
geo["densidade_demografica"] = np.where(
    geo["area"] > 0,
    geo["populacao"] / geo["area"],
    np.nan
)

# 6️⃣ Selecionar colunas corretas explicitamente
concorrente_geografia = geo[[
    "codigo_concorrente",
    "nome_conc",
    "categoria",
    "faixa_preco",
    "codigo_bairro",
    "municipio_bairro",
    "uf_bairro",
    "area",
    "populacao",
    "densidade_demografica"
]].rename(columns={
    "nome_conc": "nome_concorrente",
    "municipio_bairro": "municipio",
    "uf_bairro": "uf"
}).copy()

# 7️⃣ Ordenar
concorrente_geografia = concorrente_geografia.sort_values("codigo_concorrente")

# 8️⃣ Conferência final
print("Shape final:", concorrente_geografia.shape)
print("\nColunas finais:")
print(concorrente_geografia.columns)

concorrente_geografia.head()

Shape final: (4202, 10)

Colunas finais:
Index(['codigo_concorrente', 'nome_concorrente', 'categoria', 'faixa_preco',
       'codigo_bairro', 'municipio', 'uf', 'area', 'populacao',
       'densidade_demografica'],
      dtype='object')


,codigo_concorrente,nome_concorrente,categoria,faixa_preco,codigo_bairro,municipio,uf,area,populacao,densidade_demografica
2989,100685983380612,A Eli quem faz,Candy Store,3,3552403001,Sumaré,SP,13.3290,51200,3841.248406
3561,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,35095078,Campinas,SP,38.3206,38740,1010.944505
1980,101037933375826,Dona Formiga,Dessert Shop,0,<NA>,NaN,NaN,NaN,<NA>,NaN
886,101252426691498,Lanchonete Família Gianotti,Restaurant,2,<NA>,NaN,NaN,NaN,<NA>,NaN
502,101257903326467,Extra Itu,"Supermarket, Shopping & Retail",4,<NA>,NaN,NaN,NaN,<NA>,NaN


ENRIQUECER FLUXO COM DADOS DO CONCORRENTE

In [48]:
fluxo_final = fluxo_analitico.merge(
    faixa_preco,
    on="codigo_concorrente",
    how="left"
)
fluxo_final = fluxo_final[[
    "codigo_concorrente",
    "nome",
    "categoria",
    "faixa_preco",
    "dia_semana",
    "periodo",
    "media_fluxo",
    "max_fluxo",
    "min_fluxo",
    "total_eventos"
]]
fluxo_final

,codigo_concorrente,nome,categoria,faixa_preco,dia_semana,periodo,media_fluxo,max_fluxo,min_fluxo,total_eventos
0,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,Friday,manha,4.000000,6,3,16
1,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,Friday,noite,2.200000,3,1,11
2,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,Friday,tarde,5.250000,9,3,21
3,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,Monday,manha,3.600000,5,2,18
4,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,Monday,noite,2.600000,4,1,13
...,...,...,...,...,...,...,...,...,...,...
12267,1897386380476876,Espetinhos Do Luiz Franzoni,"Bar & Grill, Beer Garden, Product/Service",1,Thursday,tarde,1.333333,2,1,4
12268,1897386380476876,Espetinhos Do Luiz Franzoni,"Bar & Grill, Beer Garden, Product/Service",1,Tuesday,noite,1.000000,1,1,1
12269,1897386380476876,Espetinhos Do Luiz Franzoni,"Bar & Grill, Beer Garden, Product/Service",1,Tuesday,tarde,1.000000,1,1,3
12270,1897386380476876,Espetinhos Do Luiz Franzoni,"Bar & Grill, Beer Garden, Product/Service",1,Wednesday,noite,1.000000,1,1,2


Código atualizado para salvar em refined_gold

In [49]:
from pathlib import Path

# Caminho correto
OUTPUT_PATH = Path("../data/refined_gold")

# Criar pasta caso não exista
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Salvar arquivos
fluxo_final.to_parquet(
    OUTPUT_PATH / "fluxo_concorrente_analitico.parquet",
    index=False
)

concorrente_geografia.to_parquet(
    OUTPUT_PATH / "concorrente_geografia.parquet",
    index=False
)

print("Arquivos salvos com sucesso em:", OUTPUT_PATH.resolve())

Arquivos salvos com sucesso em: /home/kayo/projeto-geofusion/geofusion-case/data/refined_gold


Validar leitura da refined_gold

In [50]:

# Caminho da camada gold
GOLD_PATH = Path("../data/refined_gold")

# Ler arquivos
fluxo_gold = pd.read_parquet(GOLD_PATH / "fluxo_concorrente_analitico.parquet")
geo_gold = pd.read_parquet(GOLD_PATH / "concorrente_geografia.parquet")

# Mostrar 5 primeiros registros
print("===== FLUXO_CONCORRENTE_ANALITICO =====")
display(fluxo_gold.head())

print("\n===== CONCORRENTE_GEOGRAFIA =====")
display(geo_gold.head())

===== FLUXO_CONCORRENTE_ANALITICO =====


,codigo_concorrente,nome,categoria,faixa_preco,dia_semana,periodo,media_fluxo,max_fluxo,min_fluxo,total_eventos
0,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,Friday,manha,4.00,6,3,16
1,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,Friday,noite,2.20,3,1,11
2,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,Friday,tarde,5.25,9,3,21
3,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,Monday,manha,3.60,5,2,18
4,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,Monday,noite,2.60,4,1,13



===== CONCORRENTE_GEOGRAFIA =====


,codigo_concorrente,nome_concorrente,categoria,faixa_preco,codigo_bairro,municipio,uf,area,populacao,densidade_demografica
0,100685983380612,A Eli quem faz,Candy Store,3,3552403001,Sumaré,SP,13.3290,51200,3841.248406
1,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,35095078,Campinas,SP,38.3206,38740,1010.944505
2,101037933375826,Dona Formiga,Dessert Shop,0,<NA>,None,None,NaN,<NA>,NaN
3,101252426691498,Lanchonete Família Gianotti,Restaurant,2,<NA>,None,None,NaN,<NA>,NaN
4,101257903326467,Extra Itu,"Supermarket, Shopping & Retail",4,<NA>,None,None,NaN,<NA>,NaN


Query para para os analistas extrairem as informações

In [51]:
import duckdb


GOLD_PATH = Path("../data/refined_gold")

# Criar conexão
con = duckdb.connect()

# Registrar tabelas diretamente dos Parquets
con.execute(f"""
CREATE OR REPLACE VIEW fluxo_concorrente_analitico AS
SELECT * FROM read_parquet('{GOLD_PATH / "fluxo_concorrente_analitico.parquet"}')
""")

con.execute(f"""
CREATE OR REPLACE VIEW concorrente_geografia AS
SELECT * FROM read_parquet('{GOLD_PATH / "concorrente_geografia.parquet"}')
""")

In [55]:
fluxo_concorrente_analitico = """
SELECT 
    dia_semana,
    periodo,
    AVG(media_fluxo) AS media_geral
FROM fluxo_concorrente_analitico
GROUP BY dia_semana, periodo
ORDER BY dia_semana;
"""

resultado1 = con.execute(fluxo_concorrente_analitico).df()
resultado1.head()

,dia_semana,periodo,media_geral
0,Friday,tarde,4.854306
1,Friday,manha,3.639118
2,Friday,noite,5.119109
3,Monday,tarde,4.563923
4,Monday,manha,3.656947


In [57]:
concorrente_geografia = """
SELECT
    nome_concorrente,
    municipio,
    densidade_demografica
FROM concorrente_geografia
ORDER BY densidade_demografica DESC
LIMIT 5;
"""

resultado2 = con.execute(concorrente_geografia).df()
resultado2

,nome_concorrente,municipio,densidade_demografica
0,Pão de Açúcar - Cambuí,Campinas,10757.789237
1,Mercado Municipal De Campinas,Campinas,10757.789237
2,Giovannetti Campinas,Campinas,10757.789237
3,Pink Elephant Campinas,Campinas,10757.789237
4,Deck 21,Campinas,10757.789237


Consultar sem criar view

In [54]:
query = f"""
SELECT *
FROM read_parquet('{GOLD_PATH / "fluxo_concorrente_analitico.parquet"}')
LIMIT 5;
"""

con.execute(query).df()

,codigo_concorrente,nome,categoria,faixa_preco,dia_semana,periodo,media_fluxo,max_fluxo,min_fluxo,total_eventos
0,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,Friday,manha,4.00,6,3,16
1,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,Friday,noite,2.20,3,1,11
2,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,Friday,tarde,5.25,9,3,21
3,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,Monday,manha,3.60,5,2,18
4,101025009987066,Café e Arte,"Restaurant, Coffee Shop",2,Monday,noite,2.60,4,1,13
